In [1]:
#######################################################################
# Copyright (c) 2019-present, Blosc Development Team <blosc@blosc.org>
# All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#######################################################################

# Benchmark to compute expressions with numba and NDArray instances as operands.
# As numba takes a while to compile the first time, we use cached functions, so
# make sure to run the script at least a couple of times.

from time import time

import numba as nb
import numexpr as ne
import numpy as np

import blosc2

%load_ext cython

In [2]:
shape = (10_000, 10_000)
dtype = np.float32

# Expression to compute
exprs = (
    "x < .5",
    "(x**2 + y**2) <= (2 * x * y + 1)",
    "(sin(x)**3 + cos(y)**2) >= (cos(x) * sin(y) + z)",
)

In [3]:
# Prepare the operands
x = blosc2.linspace(0, 1, np.prod(shape), dtype=dtype, shape=shape)
y = blosc2.linspace(-1, 1, np.prod(shape), dtype=dtype, shape=shape)
z = blosc2.linspace(0, 10, np.prod(shape), dtype=dtype, shape=shape)
npx, npy, npz = x[:], y[:], z[:]
vardict = {"x": npx, "y": npy, "z": npz, "np": np}
b2vardict = {"x": x, "y": y, "z": z, "blosc2": blosc2}
[(n, a.shape, a.chunks, a.blocks) for n, a in {"x": x, "y": y, "z": z}.items()]

[('x', (10000, 10000), (400, 10000), (2, 10000)),
 ('y', (10000, 10000), (400, 10000), (2, 10000)),
 ('z', (10000, 10000), (400, 10000), (2, 10000))]

In [4]:
# Define the functions to compute the expressions

# The numba+blosc2 version using an udf
@nb.jit(parallel=True, cache=True)
def udf_numba(inputs, output, offset):
    icount = len(inputs)
    x = inputs[0]
    if icount == 1:
        for i in nb.prange(x.shape[0]):
            for j in nb.prange(x.shape[1]):
                output[i, j] = x[i, j] < 0.5
    elif icount == 2:
        y = inputs[1]
        for i in nb.prange(x.shape[0]):
            for j in nb.prange(x.shape[1]):
                output[i, j] = x[i, j] ** 2 + y[i, j] ** 2 <= 2 * x[i, j] * y[i, j] + 1
    elif icount == 3:
        y = inputs[1]
        z = inputs[2]
        for i in nb.prange(x.shape[0]):
            for j in nb.prange(x.shape[1]):
                output[i, j] = (np.sin(x[i, j]) ** 3 + np.cos(y[i, j]) ** 2) >= (
                    np.cos(x[i, j]) * np.sin(y[i, j]) + z[i, j]
                )
    return

In [5]:
# Compute expressions
for n, expr in enumerate(exprs):
    print(f"*** Computing expression: {expr} ...")

    # Compute the expression with NumPy/numexpr
    npexpr = expr.replace("sin", "np.sin").replace("cos", "np.cos")
    t0 = time()
    npres = eval(npexpr, vardict)
    print("NumPy took %.3f s" % (time() - t0))
    # ne.set_num_threads(1)
    # nb.set_num_threads(1)  # this does not work that well; better use the NUMBA_NUM_THREADS env var
    output = npres.copy()
    t0 = time()
    ne.evaluate(expr, vardict, out=output)
    print("NumExpr took %.3f s" % (time() - t0))
    # np.testing.assert_equal(output, npres)

    # Compute the expression with Blosc2
    blosc2.cparams_dflts["codec"] = blosc2.Codec.LZ4
    blosc2.cparams_dflts["clevel"] = 5
    c = blosc2.lazyexpr(expr)
    t0 = time()
    d = c.compute()
    print("LazyExpr+compute took %.3f s" % (time() - t0))
    # Check
    # np.testing.assert_equal(d[:], npres)
    t0 = time()
    d = c[:]
    print("LazyExpr+getitem took %.3f s" % (time() - t0))
    # Check
    # np.testing.assert_equal(d[:], npres)

    inputs, npinputs = (x,), (npx,)
    if n == 1:
        inputs, npinputs = (x, y), (npx, npy)
    elif n == 2:
        inputs, npinputs = (x, y, z), (npx, npy, npz)

    t0 = time()
    udf_numba(npinputs, output, offset=None)
    print("Numba took %.3f s" % (time() - t0))
    # np.testing.assert_equal(output, npres)

    expr_ = blosc2.lazyudf(udf_numba, inputs, np.bool_)
    # getitem but using chunked computation
    t0 = time()
    res = expr_.compute()
    print("LazyUDF+compute took %.3f s" % (time() - t0))
    # np.testing.assert_equal(res[...], npres)
    t0 = time()
    res = expr_[:]
    print("LazyUDF+getitem took %.3f s" % (time() - t0))
    # np.testing.assert_equal(res[...], npres)

*** Computing expression: x < .5 ...
NumPy took 0.021 s
NumExpr took 0.008 s
LazyExpr+compute took 0.027 s
LazyExpr+getitem took 0.038 s


Numba took 0.220 s
LazyUDF+compute took 0.025 s
LazyUDF+getitem took 0.029 s
*** Computing expression: (x**2 + y**2) <= (2 * x * y + 1) ...


NumPy took 0.261 s
NumExpr took 0.023 s
LazyExpr+compute took 0.061 s
LazyExpr+getitem took 0.056 s
Numba took 0.008 s
LazyUDF+compute took 0.040 s


LazyUDF+getitem took 0.036 s
*** Computing expression: (sin(x)**3 + cos(y)**2) >= (cos(x) * sin(y) + z) ...


NumPy took 0.857 s
NumExpr took 0.078 s


LazyExpr+compute took 0.152 s
LazyExpr+getitem took 0.163 s
Numba took 0.045 s


LazyUDF+compute took 0.098 s
LazyUDF+getitem took 0.090 s


In [6]:
%%cython
# The cython+blosc2 version using an udf
import numpy as np
cimport numpy as np
cimport cython
from cython.parallel cimport parallel, prange
from libc.math cimport sinf, cosf
#from cpython cimport bool
@cython.boundscheck(False)  # Deactivate bounds checking
@cython.wraparound(False)   # Deactivate negative indexing.
#def udf_cython(inputs, np.ndarray[np.npy_bool, ndim=2] output, object offset):
def udf_cython(inputs, np.npy_bool[:, ::1] output, object offset) -> None:
    cdef int icount = len(inputs)
    #print(f"*** icount: {icount}")
    cdef const np.npy_float32[:, ::1] x, y, z
    x = inputs[0]
    cdef long shape0, shape1
    shape0 = x.shape[0]
    shape1 = x.shape[1]
    cdef int i, j
    if icount == 1:
        with nogil, parallel():
            for i in prange(shape0):
                for j in prange(shape1):
                    output[i, j] = x[i, j] < .5
    elif icount == 2:
        y = inputs[1]
        with nogil, parallel():
            for i in prange(shape0):
                for j in prange(shape1):
                    output[i, j] = x[i, j]**2 + y[i, j]**2 <= 2 * x[i, j] * y[i, j] + 1
    elif icount == 3:
        y = inputs[1]
        z = inputs[2]
        with nogil, parallel():
            for i in prange(shape0):
                for j in prange(shape1):
                    output[i, j] = (sinf(x[i, j])**3 + cosf(y[i, j])**2) >= (cosf(x[i, j]) * sinf(y[i, j]) + z[i, j])
    return

In [7]:
# Compute expressions for cython
for n, expr in enumerate(exprs):
    print(f"*** Computing expression: {expr} ...")
    npres = np.empty_like(npx, dtype=np.bool_)
    ne.evaluate(expr, vardict, out=npres)

    inputs, npinputs = (x,), (npx,)
    if n == 1:
        inputs, npinputs = (x, y), (npx, npy)
    elif n == 2:
        inputs, npinputs = (x, y, z), (npx, npy, npz)

    expr_ = blosc2.lazyudf(udf_cython, inputs, np.bool_)
    # getitem but using chunked computation
    t0 = time()
    res = expr_.compute()
    print("LazyUDF+cython took %.3f s" % (time() - t0))
    # np.testing.assert_equal(res[...], npres)
    t0 = time()
    res = expr_[:]
    print("LazyUDF+getitem+cython took %.3f s" % (time() - t0))
    # np.testing.assert_equal(res[...], npres)

*** Computing expression: x < .5 ...


LazyUDF+cython took 0.029 s
LazyUDF+getitem+cython took 0.026 s
*** Computing expression: (x**2 + y**2) <= (2 * x * y + 1) ...
LazyUDF+cython took 0.060 s
LazyUDF+getitem+cython took 0.056 s
*** Computing expression: (sin(x)**3 + cos(y)**2) >= (cos(x) * sin(y) + z) ...


LazyUDF+cython took 0.661 s


LazyUDF+getitem+cython took 0.659 s


In [8]:
# The Menudet+blosc2 version


@blosc2.menudet
def udf_menudet_1(x: np.float32) -> np.bool_:
    return x < 0.5


@blosc2.menudet
def udf_menudet_2(x: np.float32, y: np.float32) -> np.bool_:
    return (x**2 + y**2) <= (2 * x * y + 1)


@blosc2.menudet
def udf_menudet_3(x: np.float32, y: np.float32, z: np.float32) -> np.bool_:
    return (np.sin(x) ** 3 + np.cos(y) ** 2) >= (np.cos(x) * np.sin(y) + z)


udf_menudet_kernels = (udf_menudet_1, udf_menudet_2, udf_menudet_3)

In [9]:
# Compute expressions for Menudet

for n, expr in enumerate(exprs):
    print(f"*** Computing expression: {expr} ...")
    npres = np.empty_like(npx, dtype=np.bool_)
    ne.evaluate(expr, vardict, out=npres)

    inputs = (x,)
    if n == 1:
        inputs = (x, y)
    elif n == 2:
        inputs = (x, y, z)

    operands = dict(zip(("x", "y", "z"), inputs))
    expr_ = udf_menudet_kernels[n].specialize(**operands).lazy(operands)
    # getitem but using chunked computation
    t0 = time()
    res = expr_.compute()
    print("Menudet+compute took %.3f s" % (time() - t0))
    # np.testing.assert_equal(res[...], npres)
    t0 = time()
    res = expr_[:]
    print("Menudet+getitem took %.3f s" % (time() - t0))
    # np.testing.assert_equal(res[...], npres)

*** Computing expression: x < .5 ...
Menudet+compute took 0.025 s
Menudet+getitem took 0.027 s
*** Computing expression: (x**2 + y**2) <= (2 * x * y + 1) ...
Menudet+compute took 0.056 s


Menudet+getitem took 0.061 s
*** Computing expression: (sin(x)**3 + cos(y)**2) >= (cos(x) * sin(y) + z) ...


Menudet+compute took 0.151 s
Menudet+getitem took 0.187 s


In [10]:
# Interactive comparison of compute speed for LazyUDF methods
import plotly.graph_objects as go

methods = ("LazyUDF+numba", "LazyUDF+cython", "Menudet")
compute_times = {method: [] for method in methods}

for n, expr in enumerate(exprs):
    inputs, npinputs = (x,), (npx,)
    if n == 1:
        inputs, npinputs = (x, y), (npx, npy)
    elif n == 2:
        inputs, npinputs = (x, y, z), (npx, npy, npz)

    output = np.empty_like(npx, dtype=np.bool_)
    udf_numba(npinputs, output, offset=None)
    expr_numba = blosc2.lazyudf(udf_numba, inputs, np.bool_)
    t0 = time()
    expr_numba.compute()
    compute_times["LazyUDF+numba"].append(time() - t0)

    expr_cython = blosc2.lazyudf(udf_cython, inputs, np.bool_)
    t0 = time()
    expr_cython.compute()
    compute_times["LazyUDF+cython"].append(time() - t0)

    operands = dict(zip(("x", "y", "z"), inputs))
    expr_menudet = udf_menudet_kernels[n].specialize(**operands).lazy(operands)
    t0 = time()
    expr_menudet.compute()
    compute_times["Menudet"].append(time() - t0)

expr_labels = [f"expr {i + 1}" for i in range(len(exprs))]
speed_ops = {m: [1.0 / t if t > 0 else float("inf") for t in ts] for m, ts in compute_times.items()}

colors = {
    "LazyUDF+numba": "#1f77b4",
    "LazyUDF+cython": "#ff7f0e",
    "Menudet": "#2ca02c",
}

fig = go.Figure()
for method in methods:
    fig.add_trace(
        go.Bar(
            x=expr_labels,
            y=speed_ops[method],
            name=method,
            marker_color=colors[method],
            customdata=compute_times[method],
            text=[f"{v:.3f}" for v in speed_ops[method]],
            textposition="outside",
            hovertemplate="Method: %{fullData.name}<br>Expression: %{x}<br>Speed: %{y:.3f} ops/s<br>Time: %{customdata:.3f} s<extra></extra>",
        )
    )

fig.update_layout(
    title="Compute speed for three expressions by LazyUDF method",
    xaxis_title="Expression",
    yaxis_title="ops/s",
    barmode="group",
    template="plotly_white",
    height=650,
)
fig.show()